# MarketScreener — Weekly Stock Screener

**Run:** Weekly, after downloading latest Yahoo Consensus CSV

**Signals:**
1. Estimate Revision Score — TP / Rev / EPS revised UP recently?
2. Price-to-TP Gap — How much upside still unpriced?
3. Price Momentum — Is price already moving in the right direction?
4. Analyst Conviction — Narrow dispersion = higher confidence
5. Combined 1/Rank Score — Same method as your existing notebooks

**Output:**
- Top 10 F&O candidates (20–25 day horizon)
- Top 10 Long-term ideas (1yr+)
- Holdings status: VENUSPIPES / ANUP / EMMVEE / ACMESOLAR

In [13]:
# ============================================================
#  CELL 1 — CONFIG
#  Edit these paths and credentials before running
# ============================================================

# --- MySQL ---
DB_HOST     = "localhost"
DB_USER     = "towdevuser"          # change if different
DB_PASSWORD = "Dev703"              # <-- TYPE YOUR PASSWORD HERE
DB_NAME     = "timelineofwealth"

# --- Excel (ReportExtract) ---
EXCEL_PATH  = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\DBInsert\ReportExtract.xlsx"
EXCEL_SHEET = "AnalystsReco"

# --- Yahoo Consensus folder ---
YAHOO_FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus"

# --- Output folder ---
OUTPUT_FOLDER = YAHOO_FOLDER   # screener output saved alongside Yahoo CSVs

# --- Screener settings ---
MIN_ANALYSTS     = 5           # minimum analyst count filter (Yahoo)
MIN_ANALYSTS_DB  = 2           # minimum broker count filter (DB/Excel)
TOP_N            = 10          # top N stocks per output list

# --- Your holdings (NSE tickers as in DB) ---
MY_HOLDINGS = ["VENUSPIPES", "ANUP", "EMMVEE", "ACMESOLAR"]

# --- Yahoo ticker → NSE ticker mapping ---
# Add entries here if Yahoo uses different symbols
YAHOO_TO_NSE = {
    # "ANUP.NS": "ANUP",
    # "EMMVEE.NS": "EMMVEE",
}

In [15]:
# ============================================================
#  CELL 2 — IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import glob
import os
from datetime import timedelta, date
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

try:
    import pymysql
    HAS_MYSQL = True
except ImportError:
    print("⚠  pymysql not installed. Install with: pip install pymysql")
    HAS_MYSQL = False

print("Imports OK")

Imports OK


In [49]:
# ============================================================
#  CELL 3 — LOAD ANALYST RECO  (DB primary, Excel fallback)
# ============================================================

# ---------- 3a. MySQL ----------
df_db = pd.DataFrame()

if HAS_MYSQL:
    try:
        conn = pymysql.connect(
            host=DB_HOST, user=DB_USER,
            password=DB_PASSWORD, database=DB_NAME
        )
        query = """
            SELECT ticker, quarter, date, mcap, cmp,
                   broker, reco, target,
                   sales_y0, sales_y1, sales_y2,
                   ebit_y0,  ebit_y1,  ebit_y2,
                   opm_y0,   opm_y1,   opm_y2,
                   roce_y0,  roce_y1,  roce_y2,
                   valuation_y0, valuation_y1, valuation_y2,
                   analyst_names
            FROM stock_analyst_reco
            WHERE date <= CURDATE()
        """
        df_db = pd.read_sql(query, conn, parse_dates=False)
        conn.close()
        df_db['date'] = pd.to_datetime(df_db['date'], errors='coerce')
        df_db['source'] = 'DB'
        print(f"✅  MySQL: {len(df_db)} rows | quarters: {sorted(df_db['quarter'].unique())}")
    except Exception as e:
        print(f"⚠  MySQL connection failed: {e}")
        print("   Will use Excel only.")

# ---------- 3b. Excel ----------
df_excel_raw = pd.read_excel(EXCEL_PATH, sheet_name=EXCEL_SHEET)
col_map = {
    'Ticker':        'ticker',
    'Quarter':       'quarter',
    'Date':          'date',
    'Mcap (Cr)':     'mcap',
    'Price':         'cmp',
    'Broker':        'broker',
    'Reco':          'reco',
    'Target':        'target',
    'Rev Y0':        'sales_y0',
    'Rev. Y1':       'sales_y1',
    'Rev. Y2':       'sales_y2',
    'EBIT Y0':       'ebit_y0',
    'EBIT Y1':       'ebit_y1',
    'EBIT Y2':       'ebit_y2',
    'EBITDA% Y0':    'opm_y0',
    'EBITDA% Y1':    'opm_y1',
    'EBITDA% Y2':    'opm_y2',
    'ROCE% Y0':      'roce_y0',
    'ROCE% Y1':      'roce_y1',
    'ROCE% Y2':      'roce_y2',
    'EV/EBIT Y0':    'valuation_y0',
    'EV/EBIT Y1':    'valuation_y1',
    'EV/EBIT Y2':    'valuation_y2',
    'Analysts Name': 'analyst_names',
    'EPS Y0':        'eps_y0',
    'EPS Y1':        'eps_y1',
    'EPS Y2':        'eps_y2',
}
df_excel_raw = df_excel_raw.rename(columns=col_map)
df_excel_raw['date'] = pd.to_datetime(df_excel_raw['date'], errors='coerce')
df_excel_raw['source'] = 'Excel'

# ---------- 3c. Merge: keep Excel only for missing quarters ----------
if not df_db.empty:
    db_quarters = set(df_db['quarter'].unique())
    df_excel_new = df_excel_raw[~df_excel_raw['quarter'].isin(db_quarters)].copy()
    print(f"   Excel new quarters (not in DB): {sorted(df_excel_new['quarter'].dropna().unique())}")
    df_reco = pd.concat([df_db, df_excel_new], ignore_index=True)
else:
    df_reco = df_excel_raw.copy()
    print("   Using Excel as sole source (no DB connection)")

# ---------- 3d. Cleanup ----------
for col in ['ticker', 'quarter', 'broker', 'reco']:
    df_reco[col] = df_reco[col].astype(str).str.strip()

for col in ['mcap', 'cmp', 'target', 'sales_y0', 'sales_y1', 'sales_y2',
            'ebit_y0', 'ebit_y1', 'ebit_y2']:
    if col in df_reco.columns:
        df_reco[col] = pd.to_numeric(df_reco[col], errors='coerce')

# Drop blank/empty rows — astype(str) converts NaN quarters to string 'nan'
df_reco = df_reco[df_reco['quarter'] != 'nan'].copy()

print(f"\n✅  Combined analyst reco: {len(df_reco)} rows, {df_reco['ticker'].nunique()} tickers")
print(f"   Quarters available: {sorted(df_reco['quarter'].unique())}")

✅  MySQL: 16012 rows | quarters: ['FY20Q1', 'FY20Q2', 'FY20Q3', 'FY20Q4', 'FY21Q1', 'FY21Q2', 'FY21Q3', 'FY21Q4', 'FY22Q1', 'FY22Q2', 'FY22Q3', 'FY22Q4', 'FY23Q1', 'FY23Q2', 'FY23Q3', 'FY23Q4', 'FY24Q1', 'FY24Q2', 'FY24Q3', 'FY24Q4', 'FY25Q1', 'FY25Q2', 'FY25Q3', 'FY25Q4', 'FY26Q1', 'FY26Q2']
   Excel new quarters (not in DB): ['FY26Q3']

✅  Combined analyst reco: 16648 rows, 535 tickers
   Quarters available: ['FY20Q1', 'FY20Q2', 'FY20Q3', 'FY20Q4', 'FY21Q1', 'FY21Q2', 'FY21Q3', 'FY21Q4', 'FY22Q1', 'FY22Q2', 'FY22Q3', 'FY22Q4', 'FY23Q1', 'FY23Q2', 'FY23Q3', 'FY23Q4', 'FY24Q1', 'FY24Q2', 'FY24Q3', 'FY24Q4', 'FY25Q1', 'FY25Q2', 'FY25Q3', 'FY25Q4', 'FY26Q1', 'FY26Q2', 'FY26Q3']


In [51]:
# ============================================================
#  CELL 4 — LOAD PRICE HISTORY FROM MySQL
# ============================================================

df_price = pd.DataFrame()
conn = pymysql.connect(
    host='localhost',
    user='towdevuser',
    password='Dev703',
    database='timelineofwealth'
)
if HAS_MYSQL:
    try:
        conn = pymysql.connect(
            host=DB_HOST, user=DB_USER,
            password=DB_PASSWORD, database=DB_NAME
        )
        price_query = """
            SELECT nse_ticker AS ticker, date, close_price
            FROM nse_price_history
            ORDER BY nse_ticker, date
        """
        df_price = pd.read_sql(price_query, conn, parse_dates=False)
        conn.close()
        df_price['date'] = pd.to_datetime(df_price['date'], errors='coerce')
        df_price['ticker'] = df_price['ticker'].astype(str).str.strip()
        print(f"✅  Price history: {len(df_price)} rows, {df_price['ticker'].nunique()} tickers")
        print(f"   Date range: {df_price['date'].min().date()} → {df_price['date'].max().date()}")
    except Exception as e:
        print(f"⚠  Price history fetch failed: {e}")
else:
    print("⚠  No MySQL — price momentum signals will be skipped")

✅  Price history: 3963162 rows, 3841 tickers
   Date range: 2015-11-27 → 2026-02-16


In [57]:
# ============================================================
#  CELL 5 — LOAD YAHOO CONSENSUS CSVs
# ============================================================
#
#  Yahoo columns used:
#    YAHOO Ticker, Date, FIN Yr
#    TP(Avg), TP(Min), TP(Max)
#    Rev.(Avg), EPS(Avg)
#    #Count, Current Price
#    TP_Appreciation

yahoo_files = sorted(glob.glob(os.path.join(YAHOO_FOLDER, "*_YahooConsensus.csv")))

if not yahoo_files:
    print("⚠  No YahooConsensus CSV files found. Yahoo signals will be skipped.")
    df_yahoo = pd.DataFrame()
else:
    dfs = []
    for f in yahoo_files:
        tmp = pd.read_csv(f)
        tmp['Date'] = pd.to_datetime(tmp['Date'])
        dfs.append(tmp)
    df_yahoo = pd.concat(dfs, ignore_index=True)

    # Apply ticker mapping (Yahoo → NSE)
    if YAHOO_TO_NSE:
        df_yahoo['NSE_Ticker'] = df_yahoo['YAHOO Ticker'].map(YAHOO_TO_NSE).fillna(df_yahoo['YAHOO Ticker'])
    else:
        df_yahoo['NSE_Ticker'] = df_yahoo['YAHOO Ticker']

    # Remove .NS suffix if present
    df_yahoo['NSE_Ticker'] = df_yahoo['NSE_Ticker'].str.replace(r'\.NS$', '', regex=True)

    for col in ['TP(Avg)', 'TP(Min)', 'TP(Max)', 'Rev.(Avg)', 'EPS(Avg)', 'Current Price']:
        if col in df_yahoo.columns:
            df_yahoo[col] = pd.to_numeric(df_yahoo[col], errors='coerce')

    df_yahoo['#Count'] = pd.to_numeric(df_yahoo.get('#Count', 0), errors='coerce').fillna(0)

    latest_yahoo_date = df_yahoo['Date'].max()
    print(f"✅  Yahoo Consensus: {len(yahoo_files)} files loaded")
    print(f"   Date range: {df_yahoo['Date'].min().date()} → {latest_yahoo_date.date()}")
    print(f"   Tickers: {sorted(df_yahoo['NSE_Ticker'].unique())}")

✅  Yahoo Consensus: 5 files loaded
   Date range: 2026-01-09 → 2026-02-13
   Tickers: ['360ONE', 'AARTIIND', 'AAVAS', 'ABFRL', 'ABSLAMC', 'ACC', 'ACMESOLAR', 'AFFLE', 'AKZOINDIA', 'ALKYLAMINE', 'AMBER', 'AMBUJACEM', 'ANANDRATHi', 'ANANTRAJ', 'ANGELONE', 'ANUP', 'APLAPOLLO', 'APOLLOHOSP', 'APTUS', 'ARVINDFASN', 'ASHOKLEY', 'ASIANPAINT', 'ASTRAL', 'AUBANK', 'AWL', 'AXISBANK', 'BAJAJ-AUTO', 'BAJAJELEC', 'BAJAJFINSV', 'BAJAJHFL', 'BAJAJHLDNG', 'BAJFINANCE', 'BANDHANBNK', 'BANKBARODA', 'BATAINDIA', 'BAYERCROP', 'BERGEPAINT', 'BHARATFORG', 'BIKAJI', 'BLUESTARCO', 'BORORENEW', 'BOSCHLTD', 'BRIGADE', 'BRITANNIA', 'BSE', 'CAMPUS', 'CAMS', 'CANBK', 'CANFINHOME', 'CARERATING', 'CARTRADE', 'CASTROLIND', 'CCL', 'CDSL', 'CELLO', 'CENTURYPLY', 'CERA', 'CHALET', 'CHOLAFIN', 'CLEAN', 'CMSINFO', 'COFORGE', 'COLPAL', 'CONCOR', 'COROMANDEL', 'CRISIL', 'CUB', 'CYIENT', 'DABUR', 'DALBHARAT', 'DCBBANK', 'DEEPAKNTR', 'DEVYANI', 'DIXON', 'DLF', 'DMART', 'DOMS', 'DREAMFOLKS', 'EASEMYTRIP', 'ECLERX', 'EDELWEISS'

In [59]:
# ============================================================
#  CELL 6 — SIGNAL 1: ESTIMATE REVISION SCORE  (Yahoo)
#
#  Compare latest Yahoo data vs 4 weeks ago:
#    TP Revision  (+/- %)
#    Rev Y1 Revision
#    EPS Y1 Revision
#  Composite = average of the three
#  Filter: #Count >= MIN_ANALYSTS
# ============================================================

df_revision_signal = pd.DataFrame()

if not df_yahoo.empty:

    def get_past_date(dates_series, weeks_back):
        target = df_yahoo['Date'].max() - timedelta(weeks=weeks_back)
        eligible = dates_series[dates_series <= target]
        return eligible.max() if not eligible.empty else None

    latest_dt   = df_yahoo['Date'].max()
    past_4w_dt  = get_past_date(df_yahoo['Date'].drop_duplicates(), 4)

    print(f"   Revision window: {past_4w_dt.date() if past_4w_dt else 'N/A'} → {latest_dt.date()}")

    if past_4w_dt is not None:
        df_L = df_yahoo[df_yahoo['Date'] == latest_dt].copy()
        df_P = df_yahoo[df_yahoo['Date'] == past_4w_dt].copy()

        # Identify Y1 (earliest FIN Yr in latest data)
        fy_sorted = sorted(df_yahoo['FIN Yr'].dropna().unique())
        Y1 = fy_sorted[0] if fy_sorted else None

        records = []
        for ticker in df_L['NSE_Ticker'].unique():
            row_L_all = df_L[df_L['NSE_Ticker'] == ticker]
            row_P_all = df_P[df_P['NSE_Ticker'] == ticker]

            if row_L_all.empty or row_P_all.empty:
                continue

            # TP revision (no FIN Yr filter)
            rL = row_L_all.iloc[0]
            rP = row_P_all.iloc[0]
            count = rL.get('#Count', 0)

            tp_rev = np.nan
            if rP.get('TP(Avg)', 0) > 0 and rL.get('TP(Avg)', 0) > 0:
                tp_rev = rL['TP(Avg)'] / rP['TP(Avg)'] - 1

            tp_appreciation = rL.get('TP_Appreciation', np.nan)
            current_price   = rL.get('Current Price',   np.nan)
            tp_avg          = rL.get('TP(Avg)',          np.nan)

            # Rev / EPS revision (Y1)
            rev_rev = np.nan
            eps_rev = np.nan
            if Y1 is not None:
                row_L_y1 = row_L_all[row_L_all['FIN Yr'] == Y1]
                row_P_y1 = row_P_all[row_P_all['FIN Yr'] == Y1]
                if not row_L_y1.empty and not row_P_y1.empty:
                    rLy = row_L_y1.iloc[0]
                    rPy = row_P_y1.iloc[0]
                    if rPy.get('Rev.(Avg)', 0) > 0 and rLy.get('Rev.(Avg)', 0) > 0:
                        rev_rev = rLy['Rev.(Avg)'] / rPy['Rev.(Avg)'] - 1
                    if rPy.get('EPS(Avg)', 0) > 0 and rLy.get('EPS(Avg)', 0) > 0:
                        eps_rev = rLy['EPS(Avg)'] / rPy['EPS(Avg)'] - 1

            # Composite revision score
            vals = [v for v in [tp_rev, rev_rev, eps_rev] if not np.isnan(v)]
            composite = np.mean(vals) if vals else np.nan

            records.append({
                'ticker':          ticker,
                'count':           count,
                'tp_avg':          tp_avg,
                'current_price':   current_price,
                'tp_appreciation': tp_appreciation,
                'tp_rev_4w':       round(tp_rev,  4) if not np.isnan(tp_rev)  else np.nan,
                'rev_rev_4w':      round(rev_rev, 4) if not np.isnan(rev_rev) else np.nan,
                'eps_rev_4w':      round(eps_rev, 4) if not np.isnan(eps_rev) else np.nan,
                'revision_score':  round(composite, 4) if not np.isnan(composite) else np.nan,
            })

        df_revision_signal = pd.DataFrame(records)
        # Apply MIN_ANALYSTS filter
        df_revision_signal = df_revision_signal[df_revision_signal['count'] >= MIN_ANALYSTS].copy()
        print(f"✅  Revision signal: {len(df_revision_signal)} tickers (>= {MIN_ANALYSTS} analysts)")
    else:
        print("⚠  Not enough historical Yahoo data for 4-week revision window")
else:
    print("⚠  Yahoo data not available — revision signal skipped")

   Revision window: 2026-01-09 → 2026-02-13
✅  Revision signal: 216 tickers (>= 5 analysts)


In [79]:
# ============================================================
#  CELL 7 — SIGNAL 2: ANALYST RECO SIGNAL  (DB + Excel)
#
#  For each ticker, take the LATEST quarter's data:
#    a. TP vs CMP gap  (upside %)
#    b. Broker consensus direction (% BUY)
#    c. Sales Y1/Y2 growth estimate
#    d. EBIT Y1/Y2 growth estimate
#  Broker count >= MIN_ANALYSTS_DB
# ============================================================

# Get latest quarter per ticker
df_reco_sorted = df_reco[df_reco['quarter'] != 'nan'].copy()
df_reco_sorted = df_reco_sorted.sort_values(['ticker', 'quarter', 'date'], ascending=[True, False, False])
latest_quarter_per_ticker = df_reco_sorted.groupby('ticker')['quarter'].first().reset_index()
latest_quarter_per_ticker.columns = ['ticker', 'latest_quarter']

df_latest = df_reco.merge(latest_quarter_per_ticker, on='ticker')
df_latest = df_latest[df_latest['quarter'] == df_latest['latest_quarter']].copy()

# Aggregrate per ticker
def reco_agg(group):
    n = len(group)
    buy_count = group['reco'].str.upper().isin(['BUY', 'STRONG BUY', 'ADD', 'ACCUMULATE']).sum()
    pct_buy = buy_count / n if n > 0 else np.nan

    cmp       = group['cmp'].median()
    tp_median = group['target'].median()
    upside    = (tp_median / cmp - 1) if cmp > 0 else np.nan

    # Sales growth Y0→Y1 and Y1→Y2
    s0 = group['sales_y0'].median()
    s1 = group['sales_y1'].median()
    s2 = group['sales_y2'].median()
    sales_g_y1 = (s1/s0 - 1) if (s0 > 0 and s1 > 0) else np.nan
    sales_g_y2 = (s2/s1 - 1) if (s1 > 0 and s2 > 0) else np.nan

    # EBIT growth Y0→Y1 and Y1→Y2
    e0 = group['ebit_y0'].median() if 'ebit_y0' in group.columns else np.nan
    e1 = group['ebit_y1'].median() if 'ebit_y1' in group.columns else np.nan
    e2 = group['ebit_y2'].median() if 'ebit_y2' in group.columns else np.nan
    ebit_g_y1 = (e1/e0 - 1) if (not np.isnan(e0) and e0 > 0 and not np.isnan(e1) and e1 > 0) else np.nan
    ebit_g_y2 = (e2/e1 - 1) if (not np.isnan(e1) and e1 > 0 and not np.isnan(e2) and e2 > 0) else np.nan

    # TP dispersion (lower = more conviction)
    if group['target'].notna().sum() >= 2:
        tp_cv = group['target'].std() / group['target'].mean()
    else:
        tp_cv = np.nan

    return pd.Series({
        'broker_count':      n,
        'latest_quarter':    group['quarter'].iloc[0],
        'cmp':               round(cmp, 2),
        'tp_median':         round(tp_median, 2) if not np.isnan(tp_median) else np.nan,
        'upside_pct':        round(upside, 4)    if not np.isnan(upside)    else np.nan,
        'pct_buy':           round(pct_buy, 4),
        'sales_g_y1':        round(sales_g_y1, 4) if not np.isnan(sales_g_y1) else np.nan,
        'sales_g_y2':        round(sales_g_y2, 4) if not np.isnan(sales_g_y2) else np.nan,
        'ebit_g_y1':         round(ebit_g_y1, 4)  if not np.isnan(ebit_g_y1)  else np.nan,
        'ebit_g_y2':         round(ebit_g_y2, 4)  if not np.isnan(ebit_g_y2)  else np.nan,
        'tp_dispersion_cv':  round(tp_cv, 4)       if not np.isnan(tp_cv)       else np.nan,
    })

df_reco_agg = df_latest.groupby('ticker').apply(reco_agg).reset_index()
df_reco_agg = df_reco_agg[df_reco_agg['broker_count'] >= MIN_ANALYSTS_DB].copy()

print(f"✅  Analyst reco signal: {len(df_reco_agg)} tickers (>= {MIN_ANALYSTS_DB} brokers)")
print(f"   Quarters represented: {sorted(df_reco_agg['latest_quarter'].unique())}")

✅  Analyst reco signal: 195 tickers (>= 2 brokers)
   Quarters represented: ['FY23Q4', 'FY24Q1', 'FY24Q3', 'FY25Q3', 'FY25Q4', 'FY26Q1', 'FY26Q2', 'FY26Q3']


In [81]:
# ============================================================
#  CELL 8 — SIGNAL 3: PRICE MOMENTUM  (MySQL price history)
# ============================================================

df_momentum = pd.DataFrame()

if not df_price.empty:
    latest_price_date = df_price['date'].max()

    def price_on(ticker_df, target_dt):
        sub = ticker_df[ticker_df['date'] <= target_dt]
        return sub.iloc[-1]['close_price'] if not sub.empty else np.nan

    windows = {
        '1W':  timedelta(weeks=1),
        '2W':  timedelta(weeks=2),
        '1M':  relativedelta(months=1),
        '3M':  relativedelta(months=3),
    }

    records = []
    for ticker, grp in df_price.groupby('ticker'):
        grp = grp.sort_values('date')
        p_now = price_on(grp, latest_price_date)
        if np.isnan(p_now) or p_now == 0:
            continue
        row = {'ticker': ticker, 'price_latest': p_now, 'price_date': latest_price_date.date()}
        for label, delta in windows.items():
            p_past = price_on(grp, latest_price_date - delta)
            row[f'ret_{label}'] = round(p_now / p_past - 1, 4) if (not np.isnan(p_past) and p_past > 0) else np.nan
        records.append(row)

    df_momentum = pd.DataFrame(records)
    print(f"✅  Price momentum: {len(df_momentum)} tickers")
    print(f"   As of: {latest_price_date.date()}")
else:
    print("⚠  Price history not available — momentum signals skipped")

✅  Price momentum: 3841 tickers
   As of: 2026-02-16


In [83]:
# ============================================================
#  CELL 9 — COMBINE ALL SIGNALS
# ============================================================
#
#  Merge on ticker (outer so nothing is dropped):
#    df_reco_agg   → anchor (all tickers with analyst coverage)
#    df_revision   → Yahoo enrichment (may be missing for some)
#    df_momentum   → price data (may be missing for some)

df_screen = df_reco_agg.copy()

if not df_revision_signal.empty:
    df_screen = df_screen.merge(
        df_revision_signal[['ticker', 'count', 'tp_avg', 'tp_appreciation',
                            'tp_rev_4w', 'rev_rev_4w', 'eps_rev_4w', 'revision_score']],
        on='ticker', how='left'
    )
    df_screen.rename(columns={'count': 'yahoo_count'}, inplace=True)

if not df_momentum.empty:
    df_screen = df_screen.merge(
        df_momentum[['ticker', 'price_latest', 'ret_1W', 'ret_2W', 'ret_1M', 'ret_3M']],
        on='ticker', how='left'
    )

# Flag thin Yahoo coverage
if 'yahoo_count' in df_screen.columns:
    df_screen['yahoo_thin'] = df_screen['yahoo_count'].fillna(0) <= 2

print(f"✅  Combined screen universe: {len(df_screen)} tickers")
print(f"   With Yahoo data:     {df_screen['revision_score'].notna().sum() if 'revision_score' in df_screen.columns else 'N/A'}")
print(f"   With price momentum: {df_screen['ret_1W'].notna().sum() if 'ret_1W' in df_screen.columns else 'N/A'}")

✅  Combined screen universe: 195 tickers
   With Yahoo data:     118
   With price momentum: 192


In [85]:
# ============================================================
#  CELL 10 — SCORE & RANK  (1/Rank composite method)
# ============================================================
#
#  Signals used for scoring:
#    A. revision_score     (Yahoo)  — higher = better
#    B. upside_pct         (DB/Exc) — higher = better
#    C. pct_buy            (DB/Exc) — higher = better
#    D. sales_g_y1         (DB/Exc) — higher = better
#    E. ebit_g_y1          (DB/Exc) — higher = better
#    F. ret_1M             (MySQL)  — moderate momentum preferred
#
#  For each signal: rank descending, compute 1/rank, sum
#
#  FNO_SCORE  = heavier weight on revision + 1M momentum
#  LT_SCORE   = heavier weight on upside + earnings growth

df_s = df_screen.copy()

def rank_signal(series, ascending=False):
    """Rank a series, return 1/rank. NaN gets the worst rank."""
    n = series.notna().sum()
    r = series.rank(ascending=ascending, na_option='bottom')
    return 1.0 / r

# ---- Build individual rank scores ----
if 'revision_score' in df_s.columns:
    df_s['rank_revision']  = rank_signal(df_s['revision_score'])
else:
    df_s['rank_revision']  = 0.0

df_s['rank_upside']        = rank_signal(df_s.get('upside_pct',        pd.Series(dtype=float)))
df_s['rank_pct_buy']       = rank_signal(df_s.get('pct_buy',           pd.Series(dtype=float)))
df_s['rank_sales_g_y1']    = rank_signal(df_s.get('sales_g_y1',        pd.Series(dtype=float)))
df_s['rank_ebit_g_y1']     = rank_signal(df_s.get('ebit_g_y1',         pd.Series(dtype=float)))

# Momentum: mild positive preferred for F&O
if 'ret_1M' in df_s.columns:
    df_s['rank_momentum']  = rank_signal(df_s['ret_1M'])
else:
    df_s['rank_momentum']  = 0.0

# ---- F&O Score (20-25 day horizon) ----
# Weights: revision=3, momentum=2, upside=1, pct_buy=1
df_s['fno_score'] = (
    3.0 * df_s['rank_revision'] +
    2.0 * df_s['rank_momentum'] +
    1.0 * df_s['rank_upside']   +
    1.0 * df_s['rank_pct_buy']
)

# ---- Long-Term Score (1yr+) ----
# Weights: upside=3, earnings_growth=2, revision=1, pct_buy=1
df_s['lt_score'] = (
    3.0 * df_s['rank_upside']     +
    2.0 * df_s['rank_ebit_g_y1']  +
    2.0 * df_s['rank_sales_g_y1'] +
    1.0 * df_s['rank_revision']   +
    1.0 * df_s['rank_pct_buy']
)

print("✅  Scores computed")
print(f"   F&O score range:  {df_s['fno_score'].min():.3f} → {df_s['fno_score'].max():.3f}")
print(f"   LT  score range:  {df_s['lt_score'].min():.3f}  → {df_s['lt_score'].max():.3f}")

✅  Scores computed
   F&O score range:  0.042 → 3.040
   LT  score range:  0.049  → 3.061


In [87]:
# ============================================================
#  CELL 11 — RESULTS: TOP 10 F&O CANDIDATES
# ============================================================

display_cols = [
    'ticker', 'latest_quarter', 'broker_count',
    'cmp', 'tp_median', 'upside_pct',
    'pct_buy',
    'revision_score', 'tp_rev_4w', 'rev_rev_4w', 'eps_rev_4w',
    'ret_1W', 'ret_2W', 'ret_1M', 'ret_3M',
    'fno_score'
]
display_cols = [c for c in display_cols if c in df_s.columns]

df_fno = df_s.sort_values('fno_score', ascending=False).head(TOP_N)[display_cols].reset_index(drop=True)
df_fno.index = df_fno.index + 1

# Flag holdings in the list
df_fno['holding'] = df_fno['ticker'].isin(MY_HOLDINGS).map({True: '⭐', False: ''})

print(f"\n{'='*70}")
print(f"  TOP {TOP_N} F&O CANDIDATES  (20-25 day horizon)")
print(f"{'='*70}")

pd.set_option('display.float_format', lambda x: f'{x:.2%}' if abs(x) < 5 else f'{x:.1f}')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 150)

print(df_fno.to_string())
print()

# Thin coverage warning
if 'yahoo_thin' in df_s.columns:
    thin = df_fno[df_fno['ticker'].isin(df_s[df_s['yahoo_thin'] == True]['ticker'])]['ticker'].tolist()
    if thin:
        print(f"⚠  Thin Yahoo coverage (<= 2 analysts): {thin} — weight consensus lightly")


  TOP 10 F&O CANDIDATES  (20-25 day horizon)
        ticker latest_quarter  broker_count    cmp  tp_median  upside_pct  pct_buy  revision_score  tp_rev_4w  rev_rev_4w  eps_rev_4w  ret_1W  ret_2W  ret_1M  ret_3M  fno_score holding
1   NAVINFLUOR         FY26Q3             2 6598.0     7225.0       9.50%   50.00%          17.54%     17.54%         NaN         NaN  -5.30%   5.89%   1.49%   3.77%    304.03%        
2     AARTIIND         FY26Q3             2  428.5      525.5      22.64%  100.00%           7.86%      7.86%         NaN         NaN  -3.22%  22.05%  28.38%  15.84%    227.79%        
3   UJJIVANSFB         FY26Q3             2   62.0       73.0      17.74%  100.00%          16.13%     16.13%         NaN         NaN  -4.81%  -0.06%   2.32%  19.19%    156.76%        
4        AMBER         FY26Q3             2 7511.0     8857.0      17.92%  100.00%           3.65%      3.65%         NaN         NaN  10.26%  29.54%  26.46%   5.36%    114.33%        
5   FEDERALBNK         FY26Q3

In [89]:
# ============================================================
#  CELL 12 — RESULTS: TOP 10 LONG-TERM IDEAS
# ============================================================

display_cols_lt = [
    'ticker', 'latest_quarter', 'broker_count',
    'cmp', 'tp_median', 'upside_pct',
    'pct_buy',
    'sales_g_y1', 'sales_g_y2',
    'ebit_g_y1',  'ebit_g_y2',
    'revision_score',
    'tp_dispersion_cv',
    'lt_score'
]
display_cols_lt = [c for c in display_cols_lt if c in df_s.columns]

df_lt = df_s.sort_values('lt_score', ascending=False).head(TOP_N)[display_cols_lt].reset_index(drop=True)
df_lt.index = df_lt.index + 1
df_lt['holding'] = df_lt['ticker'].isin(MY_HOLDINGS).map({True: '⭐', False: ''})

print(f"\n{'='*70}")
print(f"  TOP {TOP_N} LONG-TERM IDEAS  (1yr+ horizon)")
print(f"{'='*70}")
print(df_lt.to_string())
print()


  TOP 10 LONG-TERM IDEAS  (1yr+ horizon)
        ticker latest_quarter  broker_count    cmp  tp_median  upside_pct  pct_buy  sales_g_y1  sales_g_y2  ebit_g_y1  ebit_g_y2  revision_score  tp_dispersion_cv  lt_score holding
1        ABREL         FY26Q3             2 1295.0     2294.0      77.14%  100.00%      10.12%      58.69%        NaN        NaN             NaN            18.86%   306.15%        
2    SIGNATURE         FY26Q3             2  881.0     1016.5      15.38%  100.00%       7.87%      88.68%    142.89%    350.92%             NaN             0.90%   208.00%        
3      ETERNAL         FY26Q3             2  280.0      287.5       2.68%   50.00%     168.28%      75.98%        NaN        NaN          -1.84%            35.66%   204.85%        
4   NAVINFLUOR         FY26Q3             2 6598.0     7225.0       9.50%   50.00%      38.95%      24.42%    115.52%     18.47%          17.54%             8.32%   187.87%        
5     PRESTIGE         FY26Q3             3 1462.0   

In [91]:
# ============================================================
#  CELL 13 — HOLDINGS STATUS CHECK
# ============================================================
#
#  For each of the 4 holdings, show all signals side by side.
#  Traffic light:
#    🟢 upside > 20% AND pct_buy > 70% AND revision_score >= 0
#    🟡 upside > 10% OR  pct_buy > 50%
#    🔴 otherwise

print(f"\n{'='*70}")
print("  HOLDINGS STATUS")
print(f"{'='*70}")

for h in MY_HOLDINGS:
    row = df_s[df_s['ticker'] == h]
    if row.empty:
        print(f"  {h:15s}  ⚠  NOT FOUND in screener universe")
        continue
    r = row.iloc[0]

    upside   = r.get('upside_pct',     np.nan)
    pct_buy  = r.get('pct_buy',        np.nan)
    rev_sc   = r.get('revision_score', np.nan)
    ret_1m   = r.get('ret_1M',         np.nan)
    q        = r.get('latest_quarter', '?')
    brokers  = int(r.get('broker_count', 0))

    # Traffic light
    if (not np.isnan(upside)  and upside  > 0.20 and
        not np.isnan(pct_buy) and pct_buy > 0.70):
        light = '🟢'
    elif (not np.isnan(upside)  and upside  > 0.10 or
          not np.isnan(pct_buy) and pct_buy > 0.50):
        light = '🟡'
    else:
        light = '🔴'

    upside_str  = f"{upside:.1%}"  if not np.isnan(upside)  else 'N/A'
    buy_str     = f"{pct_buy:.0%}" if not np.isnan(pct_buy) else 'N/A'
    rev_str     = f"{rev_sc:+.2%}" if not np.isnan(rev_sc)  else 'N/A (no Yahoo)'
    ret1m_str   = f"{ret_1m:+.1%}" if not np.isnan(ret_1m)  else 'N/A'

    print(f"""
  {light}  {h}  [{q}]  {brokers} brokers
     Upside to TP   : {upside_str}
     % BUY reco     : {buy_str}
     Revision score : {rev_str}
     1M price return: {ret1m_str}
""")


  HOLDINGS STATUS
  VENUSPIPES       ⚠  NOT FOUND in screener universe
  ANUP             ⚠  NOT FOUND in screener universe
  EMMVEE           ⚠  NOT FOUND in screener universe
  ACMESOLAR        ⚠  NOT FOUND in screener universe


In [93]:
# ============================================================
#  CELL 14 — EXPORT TO EXCEL
# ============================================================

import os
from datetime import datetime

run_date = datetime.today().strftime('%Y%m%d')
out_file  = os.path.join(OUTPUT_FOLDER, f"{run_date}_MarketScreener.xlsx")

# Full universe with all signals
df_export = df_s.copy()
df_export['fno_rank'] = df_export['fno_score'].rank(ascending=False).astype(int)
df_export['lt_rank']  = df_export['lt_score'].rank(ascending=False).astype(int)
df_export = df_export.sort_values('fno_rank')

# Holdings
df_holdings_export = df_s[df_s['ticker'].isin(MY_HOLDINGS)].copy()

with pd.ExcelWriter(out_file, engine='xlsxwriter') as writer:
    df_fno.to_excel(writer, sheet_name='FNO_Top10',   index=True)
    df_lt.to_excel(writer,  sheet_name='LT_Top10',    index=True)
    df_holdings_export.to_excel(writer, sheet_name='Holdings',  index=False)
    df_export.to_excel(writer, sheet_name='FullUniverse', index=False)

print(f"\n✅  Screener output saved:")
print(f"   {out_file}")


✅  Screener output saved:
   C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\YahooConsensus\20260217_MarketScreener.xlsx
